# Mitra v2 (Amazon), no placar comum

Modelo de fundação tabular pré-treinado só em dados sintéticos: aprende pelo contexto, sem treinar, e pode ser ajustado (fine-tuning).
Os pesos são `autogluon/mitra-classifier-2`, pelo `autogluon.tabular` 1.6.3. O código está em `modelo.py`, com a leitura do AutoGluon que justifica cada escolha.

## Etapas

1. **Medidas** (`RODAR_MEDIDAS`): precisão na GPU, reprodutibilidade, tempo e memória. São elas que dizem quantos trials cabem.
2. **Busca** no Optuna, nas 5 dobras oficiais, com o procedimento do placar em cada dobra (`comum.estudo_placar`). Retoma de onde parou.
3. **Placar** (`RODAR_PLACAR`): trial 0, melhor trial, contexto sem oversampling e as três features, contra o trial 0 e contra o XGBoost.
4. **Teste** (`REVELAR_TESTE`): uma vez, com tudo congelado.

## Precisão

A T4 do Kaggle não tem `bfloat16` nativo, e o AutoGluon fixa essa precisão.
`PRECISAO = "auto"` usa `bfloat16` quando a GPU tem suporte nativo e `float32` quando não tem.
A etapa de medidas compara `float16` com `float32` na mesma divisão, com e sem fine-tuning.
Em `float32`, o fine-tuning na T4 pode não caber com o contexto inteiro; em `float16`, ele roda com o escalonamento do gradiente ligado, a precisão mista padrão do torch.
Se `float16` empatar com `float32` e for a única em que o fine-tuning cabe, use `PRECISAO = "float16"` na busca.

## Como executar no Kaggle

1. *Code → New Notebook → File → Import Notebook*, com o arquivo `experimentos/mitra/mitra.ipynb` da branch `BRANCH`.
2. *Add Input:* o dataset `kapturovalexander/customers-churned-in-telecom-services` (sem ele, o notebook tenta baixar pelo `kagglehub`).
3. *Settings:* acelerador **GPU T4 x2** (o notebook usa uma) e Internet **On**.
4. *Add-ons → Secrets:* `GITHUB_TOKEN`, com permissão de escrita no repositório, e `WANDB_API_KEY` (opcional), anexados ao notebook. **Nunca escreva os tokens aqui.**
5. Primeira vez: rode interativamente com `MODO_TESTE = True` e, depois, com `MODO_TESTE = False` e só as medidas (`N_TRIALS = 0`, `RODAR_PLACAR = False`).
6. Com o tempo medido, ajuste `N_TRIALS` e rode *Save Version → Save & Run All (Commit)*. Para continuar depois de 12 h, use o mesmo `RUN_NAME` e `RESUME_FROM = "github"`.

Os resultados vão para a branch `resultados`, em `<RUN_NAME>/`, a cada 30 minutos. Para trazer as linhas do Mitra para o placar do repositório: `comum.importar_placar(<pasta da execução>, "resultados", ["mitra"])`.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "rafael/mitra-kaggle"   # branch com o código
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "mitra-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = ""                    # caminho do CSV; vazio = procurar no repositório e em /kaggle/input
PRECISAO = "auto"                # "auto" | "bfloat16" | "float16" | "float32" (ver "Precisão" acima)
RODAR_MEDIDAS = True             # precisão, reprodutibilidade, tempo e memória
N_TRIALS = 100                   # trials do Optuna (o N da pré-proposta); 0 = não busca
RODAR_PLACAR = True              # trial 0, melhor trial e variantes de dados no placar
REVELAR_TESTE = False            # True só com a configuração congelada: o teste é usado uma única vez
N_SEMENTES = 5                   # sementes da avaliação final
PERMITIR_TROCAR_TORCH = False    # True deixa o pip trocar o torch do Kaggle (exige reiniciar a sessão)
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa trials novos e encerra limpo (o Kaggle corta em 12 h)
BACKUP_MIN = 30                  # envia os resultados ao GitHub a cada N minutos

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "mitra" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

# O Mitra: o pip simula antes e para se fosse trocar o torch
sys.path.insert(0, str(REPO_DIR / "experimentos" / "mitra"))
import instalar
novos = instalar.instalar(permitir_trocar_torch=PERMITIR_TROCAR_TORCH)
print("Mitra:", "instalado agora (" + ", ".join(f"{k} {v}" for k, v in sorted(novos.items())) + ")" if novos else "já instalado")

In [ ]:
import json

sys.path[:0] = [str(REPO_DIR)]
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("mitra-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if MODO_TESTE:
    N_TRIALS, N_SEMENTES = min(N_TRIALS, 2), 2
    print("MODO_TESTE: 2 trials, placar em 5 divisões e teste trocado por um pedaço do treino. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def salvar_json(nome, obj):
    json.dump(obj, open(os.path.join(PASTA, nome), "w"), indent=1, default=float)
    salvar()


def continuar():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO) > 0


import numpy as np, optuna, pandas as pd, torch
from scipy.stats import ks_2samp
import comum
import modelo as M

if not torch.cuda.is_available():
    raise RuntimeError("O Mitra só roda com GPU neste projeto. No Kaggle: Settings → Accelerator → GPU T4 x2.")
import autogluon.tabular
CAP = torch.cuda.get_device_capability(0)
print(f"GPU: {torch.cuda.get_device_name(0)} | capacidade {CAP[0]}.{CAP[1]} | {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB"
      f" | bfloat16 nativo: {M.bf16_nativo()}")
print(f"torch {torch.__version__} | autogluon.tabular {autogluon.tabular.__version__} | pandas {pd.__version__} | numpy {np.__version__}")
pd.set_option("display.max_columns", None, "display.width", 220, "display.max_colwidth", 70)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA} | CSV: {os.environ['CHURN_CSV']}")

# A linha do XGBoost vem do placar do repositório: assim o vs_xgboost aparece nas linhas do Mitra
n_xgb = comum.importar_placar(str(REPO_DIR / "resultados"), PASTA, ["xgboost"])
print("XGBoost no placar desta execução:", "sim" if any(r["modelo"] == "xgboost" for r in comum.ler_placar(PASTA)) else "não", f"({n_xgb} linhas copiadas)")

# O envio ao GitHub falhou a fase 3 do MLP inteira sem ninguém perceber: aqui ele é testado antes de qualquer treino
json.dump(dict(run=RUN_NAME, inicio=time.strftime("%Y-%m-%d %H:%M")), open(os.path.join(PASTA, "inicio.json"), "w"))
GITHUB_OK = bool(RESULTS_REPO_URL and GITHUB_TOKEN) and salvar()
if RESULTS_REPO_URL and not GITHUB_OK:
    print("=" * 90 + "\nATENÇÃO: o envio ao GitHub falhou. Troque o GITHUB_TOKEN nos Secrets ou baixe a pasta de resultados no fim.\n" + "=" * 90)
else:
    print("GitHub:", "envio testado e funcionando" if GITHUB_OK else "desligado")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="mitra", config=dict(modelo="mitra", n_trials=N_TRIALS, precisao=PRECISAO, teste=MODO_TESTE))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")

In [ ]:
# Verificações rápidas antes de gastar GPU
X_tr, X_te, y_tr, y_te = comum.dividir(comum.carregar_df())
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis com os outros modelos"
if MODO_TESTE:  # o teste de verdade nunca entra numa passada de conferência
    X_te, y_te = X_tr.iloc[:500], y_tr.iloc[:500]
ASS = comum.assinatura_treino(X_tr)
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
DIVS = comum.divisoes_placar(y_tr.values, sementes=comum.PLACAR_SEMENTES[:1] if MODO_TESTE else comum.PLACAR_SEMENTES)
for d in DIVS:
    assert not set(d["treino"]) & set(d["validacao"]) and not set(d["ajuste"]) & set(d["parada"])
print(f"teste com a assinatura oficial | KS conferido contra scipy | {len(DIVS)} divisões do placar sem sobreposição | treino {ASS}")

# 1. Medidas

As perguntas que vieram da leitura do código e que só a GPU responde, todas na primeira divisão do placar:

1. **Precisão:** `float16` dá as mesmas probabilidades que `float32`? E `bfloat16`, quando a GPU tem?
2. **Reprodutibilidade:** a mesma semente repete as probabilidades? Isso decide se 5 sementes no teste fazem sentido.
3. **Fine-tuning na GPU:** em que precisão o fine-tuning cabe com o contexto inteiro? Em `float32`, na T4, o AutoGluon chegou a cortar o contexto para 2048 linhas.
   Em `float16`, o escalonamento do gradiente fica ligado; a escala final perto de 1 indicaria passos descartados.
4. **Custo:** quanto tempo e quanta memória levam o trial 0 e o trial mais caro do espaço (8 estimadores e 100 passos de fine-tuning).

In [ ]:
def _medir(hp, precisao):
    r = M.medir_divisao(hp, X_tr, y_tr, DIVS[0], precisao)
    print(f"  {precisao:9s} KS {r['ks']:.4f} | {r['tempo_s']:.0f} s | pico {r['pico_gb']:.1f} GB", flush=True)
    return r


if RODAR_MEDIDAS and continuar():
    medidas = dict(gpu=torch.cuda.get_device_name(0), capacidade=list(CAP), bf16_nativo=M.bf16_nativo())
    print("1. Precisão, trial 0 na divisão 0")
    por_precisao = {p: _medir(M.PADRAO, p) for p in (["bfloat16"] if M.bf16_nativo() else []) + ["float16", "float32"]}
    ref = por_precisao["float32"]
    medidas["precisao"] = {p: dict(ks=r["ks"], tempo_s=r["tempo_s"], pico_gb=r["pico_gb"], delta_ks=r["ks"] - ref["ks"],
                                   max_dif_p=float(np.max(np.abs(r["p"] - ref["p"])))) for p, r in por_precisao.items()}
    display(pd.DataFrame(medidas["precisao"]).T.round(5))

    print("2. Reprodutibilidade: float32, mesma semente, duas vezes")
    outra = _medir(M.PADRAO, "float32")
    medidas["reprodutivel"] = dict(max_dif_p=float(np.max(np.abs(outra["p"] - ref["p"]))), identicas=bool(np.array_equal(outra["p"], ref["p"])))
    print("  ", medidas["reprodutivel"])

    salvar_json("medidas.json", medidas)

    print("3. Fine-tuning com 1 estimador, por precisão")
    ft = dict(M.PADRAO, fine_tune=True, lr=1e-5, fine_tune_steps=10 if MODO_TESTE else 30)
    medidas["fine_tuning"] = {}
    for p in (["bfloat16"] if M.bf16_nativo() else []) + ["float32", "float16"]:
        try:
            r = _medir(ft, p)
            medidas["fine_tuning"][p] = dict(coube=True, ks=r["ks"], tempo_s=r["tempo_s"], pico_gb=r["pico_gb"], escalas=r["escalas"])
        except RuntimeError as e:
            if "Faltou memoria" not in str(e):
                raise
            print(f"  {p:9s} não coube: {e}", flush=True)
            medidas["fine_tuning"][p] = dict(coube=False)
    display(pd.DataFrame(medidas["fine_tuning"]).T)
    salvar_json("medidas.json", medidas)

    caro = dict(M.PADRAO, n_estimators=2 if MODO_TESTE else 8, fine_tune=True, lr=1e-5, fine_tune_steps=10 if MODO_TESTE else 100)
    pedida = M.precisao_para(caro, PRECISAO)
    cabem = [p for p, r in medidas["fine_tuning"].items() if r["coube"]]
    prec_caro = pedida if pedida in cabem else (cabem[0] if cabem else None)
    if prec_caro is None:
        raise RuntimeError("O fine-tuning não coube na GPU em nenhuma precisão. Mande a saída desta célula.")
    if prec_caro != pedida:
        print(f"PRECISAO = {PRECISAO!r} ({pedida}) não coube com fine-tuning; o trial caro usa {prec_caro}.")
    print(f"4. Trial mais caro do espaço: {caro['n_estimators']} estimadores e {caro['fine_tune_steps']} passos de fine-tuning, {prec_caro}")
    try:
        r_caro = _medir(caro, prec_caro)
    except RuntimeError as e:
        if "Faltou memoria" not in str(e):
            raise
        salvar_json("medidas.json", medidas)
        raise RuntimeError(f"Com 1 estimador coube, com {caro['n_estimators']} não: {e}") from None
    medidas["caro"] = dict(hp=caro, precisao=prec_caro, ks=r_caro["ks"], tempo_s=r_caro["tempo_s"], pico_gb=r_caro["pico_gb"],
                           escalas=r_caro["escalas"])

    t_icl = medidas["precisao"][M.precisao_para(M.PADRAO, PRECISAO)]["tempo_s"]
    t_caro = r_caro["tempo_s"]
    t_medio = (t_icl + t_caro) / 2  # metade dos trials com fine-tuning, em média
    medidas["estimativa"] = dict(trial_icl_min=5 * t_icl / 60, trial_caro_min=5 * t_caro / 60, linha_placar_icl_min=15 * t_icl / 60,
                                 trials_em_10h=int(10 * 3600 / (5 * t_medio)))
    print("Estimativa:", {k: round(v, 1) for k, v in medidas["estimativa"].items()})
    f16 = medidas["precisao"]["float16"]
    print("float16 empata com float32:" if abs(f16["delta_ks"]) < 1e-3 and f16["max_dif_p"] < 1e-2 else "float16 NÃO empata com float32:",
          f"ΔKS {f16['delta_ks']:+.5f}, maior diferença de probabilidade {f16['max_dif_p']:.5f}")
    salvar_json("medidas.json", medidas)
    if wandb_run is not None:
        wandb_run.summary.update({f"medidas/{k}": v for k, v in medidas["estimativa"].items()})

# 2. Busca no Optuna

Cada trial vale o KS médio das 5 dobras oficiais (semente 42), que não são as do placar, e o corte acontece depois da segunda dobra.
O estudo fica em `<pasta>/mitra.db`: com o mesmo `RUN_NAME` e `RESUME_FROM = "github"`, a próxima sessão continua dele.
O limite de horas só é conferido entre trials.

In [ ]:
PRECISAO_ESTUDO = PRECISAO
if N_TRIALS > 0 and continuar():
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    cb = fluxo.callback_trial(wandb_run, salvar, BACKUP_MIN)
    st = comum.estudo_placar(M.MODELO, N_TRIALS, M.sugerir, lambda hp: M.ajustar_prever_de(hp, precisao=PRECISAO_ESTUDO), M.PADRAO,
                             X_tr, y_tr, pasta=PASTA, callbacks=[cb], continuar=continuar)
    salvar()
    ok = [t for t in st.trials if t.state.name == "COMPLETE"]
    print(f"{len(ok)} completos, {len([t for t in st.trials if t.state.name == 'PRUNED'])} podados | "
          f"{st.user_attrs.get('tempo_parede_s', 0) / 3600:.1f} h de parede no total")
    if ok:
        print(f"KS trial 0 = {st.trials[0].value:.4f} -> melhor = {st.best_value:.4f} (trial {st.best_trial.number})", st.best_params)
        linhas = [dict(hiperparametro=h, valor=str(t.params[h]), ks=t.value) for t in ok for h in t.params
                  if h not in ("lr", "fine_tune_steps")]
        display(pd.DataFrame(linhas).groupby(["hiperparametro", "valor"]).ks.agg(["count", "max", "median"]).round(4))
        try:
            print("Importâncias:", {k: round(v, 3) for k, v in optuna.importance.get_param_importances(st).items()})
        except Exception as e:
            print("Importâncias indisponíveis:", e)

# 3. Placar

Cada linha são 15 ajustes (3 repetições × 5 dobras) que a busca não viu.
`vs_trial0` responde quanto a busca ajudou; `vs_xgboost` entra sozinho quando a linha do XGBoost está no placar desta execução.
As variantes de dados seguem o `docs/GUIA_DADOS.md`: o contexto sem oversampling e as três features.
O final só troca o melhor trial por uma variante que **ganhou** dele; empate não troca.

In [ ]:
def config(hp, features=()):
    return dict(hp=dict(hp), features=list(features))


def garantir(rotulo, cfg, contra=None):
    def medir():
        f = M.ajustar_prever_de(cfg["hp"], cfg["features"], PRECISAO)
        t0 = time.time()
        r = comum.medir_placar(X_tr, y_tr, f, divs=DIVS)
        r.update(precisao=f.precisao, tempo_s=time.time() - t0)
        return r
    linha, _ = comum.garantir_placar(PASTA, M.MODELO, rotulo, cfg, medir, y_tr, ASS, contra)
    salvar()
    return linha


def mostrar(linhas, campos):
    rows = []
    for r in linhas:
        row = dict(rotulo=r["rotulo"], ks_kfold=r["ks_kfold"], ep=r["ep"], ks_oof=r["ks_oof"], auc_oof=r["auc_oof"], mse_val=r["mse_val"],
                   ce_val=r["ce_val"], precisao=r.get("precisao"), min=r.get("tempo_s", 0) / 60)
        for c in campos:
            v = r.get(c)
            row[c] = "" if not v else f"{v['veredito']} {v['delta']:+.4f} [{v['ic95'][0]:+.4f}, {v['ic95'][1]:+.4f}]"
        rows.append(row)
    display(pd.DataFrame(rows).round(4))


T0, OPT, TRES = (M.MODELO, "trial0"), (M.MODELO, "optuna"), ("cliente_novo", "delta_preco", "tenure_1")
CFG_FINAL = None
if RODAR_PLACAR and continuar():
    linhas = [garantir("trial0", config(M.PADRAO))]
    st = optuna.load_study(study_name=M.MODELO, storage=f"sqlite:///{PASTA}/{M.MODELO}.db") if os.path.isfile(f"{PASTA}/{M.MODELO}.db") else None
    if st is not None and any(t.state.name == "COMPLETE" for t in st.trials) and continuar():
        HP_OPT = M.sugerir(optuna.trial.FixedTrial(st.best_params))
        linhas.append(garantir("optuna", config(HP_OPT), {"vs_trial0": T0}))
        variantes = {"sem_over" if HP_OPT["over"] else "com_over": config(dict(HP_OPT, over=not HP_OPT["over"])),
                     "tres_features": config(HP_OPT, TRES)}
        linhas_var = {}
        for rot, cfg in variantes.items():
            if continuar():
                linhas_var[rot] = garantir(rot, cfg, {"vs_optuna": OPT, "vs_trial0": T0})
        linhas += list(linhas_var.values())
        ganharam = {r: l for r, l in linhas_var.items() if (l.get("vs_optuna") or {}).get("veredito") == "ganhou"}
        rot_final = max(ganharam, key=lambda r: ganharam[r]["ks_oof"]) if ganharam else "optuna"
        CFG_FINAL = variantes[rot_final] if ganharam else config(HP_OPT)
        motivo = f"{rot_final} ganhou do melhor trial no placar" if ganharam else "melhor trial do Optuna; nenhuma variante de dados ganhou dele"
        if len(linhas_var) == len(variantes) and continuar():
            linhas.append(garantir("final", CFG_FINAL, {"vs_trial0": T0}))
            comum.adotar_placar(PASTA, M.MODELO, "final", motivo)
            print("final =", rot_final, "|", motivo, "|", json.dumps(CFG_FINAL))
    mostrar(linhas, ["vs_trial0", "vs_optuna", "vs_xgboost"])

# 4. Teste

Só com `REVELAR_TESTE = True`, uma vez, com a linha `final` no placar.
O limiar das métricas de classificação vem da OOF do placar, nunca do teste.

In [ ]:
if REVELAR_TESTE and CFG_FINAL is not None and continuar():
    fin = M.final(CFG_FINAL["hp"], X_tr, y_tr, X_te, N_SEMENTES, CFG_FINAL["features"], PRECISAO)
    comum.registrar_teste_placar(PASTA, M.MODELO, "final", y_te.values, fin["p_sementes"])
    linha_final = {r["rotulo"]: r for r in comum.ler_placar(PASTA) if r["modelo"] == M.MODELO}["final"]
    limiar = comum.limiar_ks(y_tr.values, np.load(os.path.join(PASTA, linha_final["oof"])).mean(0))
    p = np.mean(fin["p_sementes"], 0)
    m = comum.metricas(y_te.values, p, limiar)
    m.update(ks_ic95=comum.bootstrap_ks(y_te.values, p), ks_por_semente=[comum.ks(y_te.values, np.asarray(x)) for x in fin["p_sementes"]])
    salvar_json("teste.json", dict(config=CFG_FINAL, teste=m, **fin, y_teste=[int(v) for v in y_te]))
    print(json.dumps({k: v for k, v in m.items() if k != "confusao"}, indent=1)); print("confusão:", m["confusao"])
else:
    print("Teste não revelado.")

In [ ]:
salvar()
if wandb_run is not None:
    wandb_run.finish()
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))
print(f"{(time.time() - INICIO) / 3600:.2f} h nesta sessão")